# LeNet-5 Neural Network for MNIST

### Options

In [1]:
batch_size = 20
training_epochs = 10

### Load & convert data

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def convert_data(x_data, y_data):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    return (x_data, y_data)

# for now we are just interested in the training data and not the test data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)
(X_test, y_test) = convert_data(X_test, y_test)

2024-10-28 13:14:33.298631: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1730117673.381329  158733 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1730117673.417231  158733 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-28 13:14:33.597659: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
print("Shapes:")
print((X_train.shape, y_train.shape))

Shapes:
((60000, 32, 32, 1), (60000, 10))


### Setup our model

In [4]:
model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 1)),
    keras.layers.Conv2D(6, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(16, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(120),
    keras.layers.Dense(84),
    keras.layers.Dense(10),
    keras.layers.Activation('softmax')
])

model.summary()

2024-10-28 13:14:39.060714: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Fitting the model

In [5]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

Epoch 1/10


2024-10-28 13:14:39.463808: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 196608000 exceeds 10% of free system memory.


2400/2400 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.8530 - loss: 1.9024 - val_accuracy: 0.9570 - val_loss: 0.1485
Epoch 2/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9606 - loss: 0.1338 - val_accuracy: 0.9688 - val_loss: 0.0960
Epoch 3/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9661 - loss: 0.1148 - val_accuracy: 0.9653 - val_loss: 0.1122
Epoch 4/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9697 - loss: 0.1022 - val_accuracy: 0.9715 - val_loss: 0.0881
Epoch 5/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9732 - loss: 0.0895 - val_accuracy: 0.9747 - val_loss: 0.0911
Epoch 6/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9771 - loss: 0.0757 - val_accuracy: 0.9776 - val_loss: 0.0713
Epoch 7/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9772 - loss: 0.0724 - val_accuracy: 0.9803 - val_loss: 0.0710
Epoch 8/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9788 - loss: 0.0697 - val

## How good is our model for non-train data?

In [6]:
score = model.evaluate(X_test, y_test, batch_size=batch_size)

score

500/500 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9727 - loss: 0.1117


[0.09370552003383636, 0.9768000245094299]

In [18]:
model.export("export")

!python -m tf2onnx.convert --saved-model export --output mnist.onnx

INFO:tensorflow:Assets written to: export/assets


INFO:tensorflow:Assets written to: export/assets


Saved artifact at 'export'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  129603780705360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603780705744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778200784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778201744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778201360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778201936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778199632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778202320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778200208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  129603778202704: TensorSpec(shape=(), dtype=tf.resource, name=None)
E0000 00:00:1730118552.1656